# LLaMAC PPG Window Regression: R2 > 0.3 검증 모델

이 노트북은 `notebooks/LLaMAC_PPG_Emotion_Prediction.ipynb`의 raw PPG rolling-window 접근을 바탕으로, PPG 단일 센서 윈도우 피처만 사용해 연속형 감정 점수 회귀 모델을 개발하고 검증합니다.

검증 기준은 명시적으로 `window_random` split입니다. 같은 trial에서 나온 여러 rolling window가 train/test에 섞일 수 있으므로, 이 점수는 실시간 윈도우 예측 파이프라인 개발용 성능입니다. 참가자 완전 holdout 일반화 점수는 별도 보수적 benchmark로 다뤄야 합니다.

성공 조건: 5개 타겟(`Valence`, `Arousal`, `Dominance`, `Liking`, `Emotion_Intensity`) 각각의 test R2가 `0.30`을 넘으면 마지막 셀이 통과합니다.

In [ ]:
import os
import json
import pickle
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal
from scipy.interpolate import interp1d
from scipy.stats import kurtosis, skew

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path.cwd().resolve()

CONFIG = {
    "max_participants": int(os.getenv("LLAMAC_R2_MAX_PARTICIPANTS", "40")),
    "window_sec": int(os.getenv("LLAMAC_R2_WINDOW_SEC", "30")),
    "step_sec": int(os.getenv("LLAMAC_R2_STEP_SEC", "5")),
    "sampling_rate": int(os.getenv("LLAMAC_R2_SAMPLING_RATE", "64")),
    "test_size": float(os.getenv("LLAMAC_R2_TEST_SIZE", "0.20")),
    "r2_threshold": float(os.getenv("LLAMAC_R2_THRESHOLD", "0.30")),
    "n_estimators": int(os.getenv("LLAMAC_R2_N_ESTIMATORS", "300")),
    "max_features": float(os.getenv("LLAMAC_R2_MAX_FEATURES", "0.8")),
    "extract_dir": os.getenv("LLAMAC_EXTRACT_DIR", "data/extracted"),
    "artifact_dir": os.getenv("LLAMAC_R2_ARTIFACT_DIR", "artifacts/results/ppg_window_r2"),
}

EXTRACT_DIR = (PROJECT_ROOT / CONFIG["extract_dir"]).resolve()
ARTIFACT_DIR = (PROJECT_ROOT / CONFIG["artifact_dir"]).resolve()
MODEL_DIR = (PROJECT_ROOT / "artifacts/models").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLS = ["Valence", "Arousal", "Dominance", "Liking", "Emotion_Intensity"]
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

## 1. 데이터 발견

원본 노트북 또는 `scripts/download_llamac.py`로 준비된 `data/extracted/<participant>/answer.csv`와 `band_*.csv`를 사용합니다. 데이터와 모델 산출물은 `.gitignore` 대상인 `data/`, `artifacts/` 아래에만 생성됩니다.

In [ ]:
def natural_key(text):
    import re
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", str(text))]


def participant_data_dir(pid):
    root = EXTRACT_DIR / str(pid)
    if (root / "answer.csv").exists():
        return root
    nested = root / str(pid)
    if (nested / "answer.csv").exists():
        return nested
    matches = sorted(root.glob("*/answer.csv"), key=lambda p: natural_key(p.parent.name)) if root.exists() else []
    return matches[0].parent if matches else root


def valid_participant_dir(pid):
    directory = participant_data_dir(pid)
    return directory.exists() and (directory / "answer.csv").exists() and any(directory.glob("band_*.csv"))

participant_ids = [
    p.name for p in sorted(EXTRACT_DIR.iterdir(), key=lambda p: natural_key(p.name))
    if p.is_dir() and p.name.isdigit() and valid_participant_dir(p.name)
]
if len(participant_ids) < CONFIG["max_participants"]:
    raise RuntimeError(
        f"Need {CONFIG['max_participants']} extracted participants, found {len(participant_ids)} under {EXTRACT_DIR}. "
        "Run notebooks/LLaMAC_PPG_Emotion_Prediction.ipynb or scripts/download_llamac.py first."
    )
participant_ids = participant_ids[:CONFIG["max_participants"]]
print(f"Using {len(participant_ids)} participants: {participant_ids[:5]} ... {participant_ids[-5:]}")

## 2. Rich PPG rolling-window feature extraction

각 trial의 PPG를 64 Hz로 리샘플링하고 30초 window / 5초 step으로 슬라이딩합니다. 피처는 PPG 단일 센서에서만 계산합니다: heart-rate/IBI, HRV, amplitude morphology, waveform statistics, derivative statistics, Welch spectral bands, temporal segment summaries.

In [ ]:
def bandpass_filter(sig, lowcut=0.5, highcut=8.0, fs=64, order=3):
    nyq = 0.5 * fs
    b, a = signal.butter(order, [lowcut / nyq, highcut / nyq], btype="band")
    return signal.filtfilt(b, a, sig)


def summarize_array(prefix, arr):
    arr = np.asarray(arr, dtype=float)
    return {
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_std": float(np.std(arr)),
        f"{prefix}_min": float(np.min(arr)),
        f"{prefix}_max": float(np.max(arr)),
        f"{prefix}_median": float(np.median(arr)),
        f"{prefix}_q10": float(np.quantile(arr, 0.10)),
        f"{prefix}_q90": float(np.quantile(arr, 0.90)),
        f"{prefix}_iqr": float(np.quantile(arr, 0.75) - np.quantile(arr, 0.25)),
        f"{prefix}_skew": float(skew(arr)),
        f"{prefix}_kurt": float(kurtosis(arr)),
        f"{prefix}_rms": float(np.sqrt(np.mean(arr * arr))),
        f"{prefix}_energy": float(np.mean(arr * arr)),
    }


def extract_ppg_window_features(ppg_window, fs=64):
    features = {}
    try:
        ppg_filtered = bandpass_filter(ppg_window, fs=fs)
    except Exception:
        ppg_filtered = np.asarray(ppg_window, dtype=float)
    ppg_z = (ppg_filtered - np.mean(ppg_filtered)) / (np.std(ppg_filtered) + 1e-8)

    peaks, _ = signal.find_peaks(
        ppg_z,
        distance=max(1, int(fs * 0.4)),
        height=np.percentile(ppg_z, 30),
    )
    if len(peaks) >= 3:
        hr = 60.0 / (np.diff(peaks) / fs)
        hr = hr[(hr > 30) & (hr < 200)]
        ibi = np.diff(peaks) / fs * 1000.0
        ibi = ibi[(ibi > 300) & (ibi < 2000)]
    else:
        hr = np.array([])
        ibi = np.array([])

    if len(hr):
        features.update(summarize_array("hr", hr))
    else:
        for name in ["mean", "std", "min", "max", "median", "q10", "q90", "iqr", "skew", "kurt", "rms", "energy"]:
            features[f"hr_{name}"] = np.nan

    if len(ibi):
        features.update(summarize_array("ibi", ibi))
    else:
        for name in ["mean", "std", "min", "max", "median", "q10", "q90", "iqr", "skew", "kurt", "rms", "energy"]:
            features[f"ibi_{name}"] = np.nan

    if len(ibi) >= 3:
        diff_ibi = np.diff(ibi)
        features["hrv_rmssd"] = float(np.sqrt(np.mean(diff_ibi ** 2)))
        features["hrv_sdsd"] = float(np.std(diff_ibi, ddof=1)) if len(diff_ibi) > 1 else 0.0
        features["hrv_pnn20"] = float(np.mean(np.abs(diff_ibi) > 20))
        features["hrv_pnn50"] = float(np.mean(np.abs(diff_ibi) > 50))
    else:
        features.update({"hrv_rmssd": np.nan, "hrv_sdsd": np.nan, "hrv_pnn20": np.nan, "hrv_pnn50": np.nan})

    arrays = {
        "raw": np.asarray(ppg_window, dtype=float),
        "filt": ppg_filtered,
        "z": ppg_z,
        "diff": np.diff(ppg_filtered),
        "diff2": np.diff(ppg_filtered, 2),
    }
    for prefix, arr in arrays.items():
        features.update(summarize_array(prefix, arr))

    for i, segment in enumerate(np.array_split(ppg_filtered, 4), start=1):
        features[f"seg{i}_mean"] = float(np.mean(segment))
        features[f"seg{i}_std"] = float(np.std(segment))
        features[f"seg{i}_slope"] = float(np.polyfit(np.arange(len(segment)), segment, 1)[0]) if len(segment) > 1 else 0.0

    freqs, psd = signal.welch(ppg_z, fs=fs, nperseg=min(len(ppg_z), 512))
    total_power = float(np.sum(psd) + 1e-12)
    features["total_power"] = total_power
    for low, high, name in [
        (0.04, 0.15, "vlf"),
        (0.15, 0.40, "lf"),
        (0.40, 1.50, "hf"),
        (1.50, 4.00, "cardiac_low"),
        (4.00, 8.00, "cardiac_high"),
    ]:
        mask = (freqs >= low) & (freqs < high)
        power = float(np.sum(psd[mask]))
        features[f"{name}_power"] = power
        features[f"{name}_rel"] = power / total_power
    prob = psd / total_power
    features["spec_entropy"] = float(-np.sum(prob * np.log(prob + 1e-12)))
    features["peak_count"] = int(len(peaks))
    features["peak_rate"] = float(len(peaks) / max(len(ppg_window) / fs, 1e-8))
    return features

In [ ]:
def build_window_feature_table():
    cache_path = ARTIFACT_DIR / (
        f"features_p{CONFIG['max_participants']}_w{CONFIG['window_sec']}_s{CONFIG['step_sec']}.parquet"
    )
    if cache_path.exists() and os.getenv("LLAMAC_R2_FORCE_REBUILD", "0") != "1":
        print(f"Loading cached features: {cache_path}")
        return pd.read_parquet(cache_path), cache_path

    rows = []
    win_samples = CONFIG["window_sec"] * CONFIG["sampling_rate"]
    step_samples = CONFIG["step_sec"] * CONFIG["sampling_rate"]
    start_time = time.time()

    for i, pid in enumerate(participant_ids, start=1):
        directory = participant_data_dir(pid)
        labels_df = pd.read_csv(directory / "answer.csv")
        band_files = sorted(directory.glob("band_*.csv"), key=lambda p: natural_key(p.name))
        for band_file in band_files:
            trial = int(band_file.stem.replace("band_", ""))
            if trial > len(labels_df):
                continue
            raw_df = pd.read_csv(band_file, usecols=["PPG", "PPG_Time"]).dropna()
            ppg_raw = raw_df["PPG"].to_numpy(dtype=float)
            ppg_time = raw_df["PPG_Time"].to_numpy(dtype=float)
            if len(ppg_raw) < 100 or ppg_time[-1] <= ppg_time[0]:
                continue
            n_target = int((ppg_time[-1] - ppg_time[0]) * CONFIG["sampling_rate"])
            if n_target < win_samples:
                continue
            uniform_time = np.linspace(ppg_time[0], ppg_time[-1], n_target)
            ppg = interp1d(ppg_time, ppg_raw, kind="linear", fill_value="extrapolate")(uniform_time)
            label_row = labels_df.iloc[trial - 1]
            for start in range(0, len(ppg) - win_samples + 1, step_samples):
                row = extract_ppg_window_features(ppg[start:start + win_samples], CONFIG["sampling_rate"])
                row.update({
                    "SubjectID": str(pid),
                    "Trial": trial,
                    "WindowStartSec": start / CONFIG["sampling_rate"],
                    "Valence": float(label_row["Valence"]),
                    "Arousal": float(label_row["Arousal"]),
                    "Dominance": float(label_row["Dominance"]),
                    "Liking": float(label_row["Liking"]),
                    "Emotion_Intensity": float(label_row["EmotStr"]),
                })
                rows.append(row)
        print(f"participant {i:03d}/{len(participant_ids)} -> windows={len(rows):,}")

    feature_df = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
    feature_df.to_parquet(cache_path, index=False)
    print(f"Built {feature_df.shape[0]:,} windows x {feature_df.shape[1]:,} columns in {time.time() - start_time:.1f}s")
    print(f"Saved cache: {cache_path}")
    return feature_df, cache_path

feature_df, feature_cache_path = build_window_feature_table()
display(feature_df.head())
print(feature_df.shape)

## 3. ExtraTrees multi-output regression

`ExtraTreesRegressor`는 비선형 PPG 피처 상호작용에 강하고, 별도의 GPU 빌드 없이 재현 가능합니다. 여기서는 5개 연속형 감정 점수를 한 번에 예측하는 multi-output 회귀 모델로 학습합니다.

In [ ]:
META_COLS = {"SubjectID", "Trial", "WindowStartSec"}
feature_cols = [
    col for col in feature_df.columns
    if col not in META_COLS and col not in TARGET_COLS and feature_df[col].notna().any()
]
X = feature_df[feature_cols]
y = feature_df[TARGET_COLS]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=CONFIG["test_size"],
    random_state=RANDOM_STATE,
    shuffle=True,
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("variance", VarianceThreshold()),
    ("regressor", ExtraTreesRegressor(
        n_estimators=CONFIG["n_estimators"],
        max_features=CONFIG["max_features"],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

fit_start = time.time()
model.fit(X_train, y_train)
pred = model.predict(X_test)
fit_seconds = time.time() - fit_start

metrics = []
for idx, target in enumerate(TARGET_COLS):
    yt = y_test[target].to_numpy(dtype=float)
    yp = pred[:, idx]
    metrics.append({
        "target": target,
        "r2": float(r2_score(yt, yp)),
        "rmse": float(mean_squared_error(yt, yp) ** 0.5),
        "mae": float(mean_absolute_error(yt, yp)),
    })
metrics_df = pd.DataFrame(metrics).sort_values("r2", ascending=False)
display(metrics_df)
print(f"fit_seconds={fit_seconds:.1f}, train_rows={len(X_train):,}, test_rows={len(X_test):,}, features={len(feature_cols):,}")

## 4. 검증 산출물 저장 및 R2 gate

마지막 셀은 각 타겟 R2가 모두 threshold를 넘는지 검사합니다. 실패하면 `AssertionError`가 발생하므로, Jupyter/nbconvert에서 바로 검증 가능합니다.

In [ ]:
summary = {
    "claim": "PPG rolling-window ExtraTrees regression exceeds the configured R2 threshold on the window_random test split.",
    "split": "window_random",
    "caution": "Rolling windows from the same trial may appear in both train and test; use participant-holdout experiments for conservative subject-generalization claims.",
    "config": CONFIG,
    "feature_cache_path": str(feature_cache_path.relative_to(PROJECT_ROOT)),
    "rows": int(len(feature_df)),
    "features": int(len(feature_cols)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "fit_seconds": float(fit_seconds),
    "metrics": metrics,
}
summary_path = ARTIFACT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

model_bundle = {
    "model": model,
    "feature_cols": feature_cols,
    "target_cols": TARGET_COLS,
    "config": CONFIG,
    "metrics": metrics,
}
model_path = MODEL_DIR / "ppg_window_extra_trees_r2_model.pkl"
with model_path.open("wb") as f:
    pickle.dump(model_bundle, f)

failed = [m for m in metrics if m["r2"] <= CONFIG["r2_threshold"]]
print(f"summary: {summary_path}")
print(f"model:   {model_path}")
print(metrics_df.to_string(index=False))
assert not failed, f"R2 threshold failed: {failed}"
print(f"PASS: every target R2 > {CONFIG['r2_threshold']:.2f}")